# Study 1 — 01b: Label Distribution Analysis

Analyses marginal label distributions overall, by task, by set, and by trace completion status.

**Outputs:**
- `outputs/study1_analysis/tables/label_distribution_*.csv`
- `outputs/study1_analysis/tables/flag_distributions.csv`
- `outputs/study1_analysis/figures/label_distribution_micro.png`
- `outputs/study1_analysis/figures/label_distribution_by_task.png`

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
from study1_helpers import *

traces = load_traces()
df = build_sentence_df(traces)
print(f'Loaded: {df.trace_key.nunique()} traces, {len(df):,} sentences')

## 2a. Overall Marginal Distribution

In [ ]:
section_header('2a. Overall Marginal Distribution')

# Micro-label counts and percentages
micro_dist = df['micro_label'].value_counts().reindex(MICRO_LABELS)
micro_pct = (micro_dist / len(df) * 100).round(2)
dist_df = pd.DataFrame({'count': micro_dist, 'pct': micro_pct, 'macro': [MACRO_MAP[m] for m in MICRO_LABELS]})
print('Micro-label distribution:')
display(dist_df)

# Macro-label counts
macro_dist = df['macro_label'].value_counts().reindex(MACRO_LABELS)
macro_pct = (macro_dist / len(df) * 100).round(2)
macro_df = pd.DataFrame({'count': macro_dist, 'pct': macro_pct})
print('\nMacro-label distribution:')
display(macro_df)

save_table(dist_df, 'label_distribution_overall.csv')

# Visualise: bar chart coloured by macro
fig, ax = plt.subplots(figsize=(10, 5))
colours = [MICRO_COLOURS[m] for m in MICRO_LABELS]
ax.bar(MICRO_LABELS, micro_dist.values, color=colours, edgecolor='white')
ax.set_ylabel('Count')
ax.set_xlabel('Micro-label')
ax.set_title('Overall Micro-label Distribution')

# Legend for macro colours
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=MACRO_COLOURS[m], label=m) for m in MACRO_LABELS]
ax.legend(handles=legend_elements, title='Macro-label', loc='upper right')
ax.set_xticklabels(MICRO_LABELS, rotation=45, ha='right')
plt.tight_layout()
save_fig(fig, 'label_distribution_micro.png')
plt.show()

## 2b. Distribution by Task

In [ ]:
section_header('2b. Distribution by Task')

# Micro-label proportions per task
task_dist = df.groupby(['task_id', 'micro_label']).size().unstack(fill_value=0)
task_dist = task_dist.reindex(columns=MICRO_LABELS)
task_pct = task_dist.div(task_dist.sum(axis=1), axis=0) * 100

print('Micro-label proportions (%) by task:')
display(task_pct.round(1))

save_table(task_pct, 'label_distribution_by_task.csv')

# Visualise: grouped bar chart
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(MICRO_LABELS))
width = 0.2
for i, task in enumerate(sorted(df['task_id'].unique())):
    vals = task_pct.loc[task].values
    ax.bar(x + i * width, vals, width, label=f'Task {task}', alpha=0.85)

ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(MICRO_LABELS, rotation=45, ha='right')
ax.set_ylabel('Percentage of sentences')
ax.set_title('Micro-label Distribution by Task')
ax.legend(title='Task')
plt.tight_layout()
save_fig(fig, 'label_distribution_by_task.png')
plt.show()

## 2c. Distribution by Set

In [ ]:
section_header('2c. Distribution by Set')

set_dist = df.groupby(['set', 'micro_label']).size().unstack(fill_value=0)
set_dist = set_dist.reindex(columns=MICRO_LABELS)
set_pct = set_dist.div(set_dist.sum(axis=1), axis=0) * 100

print('Micro-label proportions (%) by set:')
display(set_pct.round(1))

# Chi-squared test
from scipy.stats import chi2_contingency
chi2, p, dof, expected = chi2_contingency(set_dist)
print(f'\nChi-squared test (Set A vs Set B): χ²={chi2:.2f}, p={p:.4f}, dof={dof}')

save_table(set_pct, 'label_distribution_by_set.csv')

## 2d. Distribution by Completion Status

In [ ]:
section_header('2d. Distribution by Completion Status')

comp_dist = df.groupby(['completed', 'micro_label']).size().unstack(fill_value=0)
comp_dist = comp_dist.reindex(columns=MICRO_LABELS)
comp_pct = comp_dist.div(comp_dist.sum(axis=1), axis=0) * 100
comp_pct.index = ['Truncated', 'Completed']

print('Micro-label proportions (%) by completion status:')
display(comp_pct.round(1))

# Key comparison: RULE and TEST
print(f'\nRULE: completed={comp_pct.loc["Completed", "RULE"]:.1f}%, truncated={comp_pct.loc["Truncated", "RULE"]:.1f}%')
print(f'TEST: completed={comp_pct.loc["Completed", "TEST"]:.1f}%, truncated={comp_pct.loc["Truncated", "TEST"]:.1f}%')

save_table(comp_pct, 'label_distribution_by_completion.csv')

## 2e. Flag Distributions

In [ ]:
section_header('2e. Flag Distributions')

# test_context
test_df = df[df['micro_label'] == 'TEST']
tc_dist = test_df['test_context'].value_counts()
tc_pct = (tc_dist / len(test_df) * 100).round(1)
print('test_context distribution:')
for k, v in tc_pct.items():
    print(f'  {k}: {tc_dist[k]} ({v}%)')

# specificity
sp_dist = test_df['specificity'].value_counts()
sp_pct = (sp_dist / len(test_df) * 100).round(1)
print('\nspecificity distribution:')
for k, v in sp_pct.items():
    print(f'  {k}: {sp_dist[k]} ({v}%)')

# judgement
judge_df = df[df['micro_label'] == 'JUDGE']
j_dist = judge_df['judgement'].value_counts()
j_pct = (j_dist / len(judge_df) * 100).round(1)
print('\njudgement distribution:')
for k, v in j_pct.items():
    print(f'  {k}: {j_dist[k]} ({v}%)')

# Save combined flag table
flag_rows = []
for k, v in tc_dist.items():
    flag_rows.append({'flag': 'test_context', 'value': k, 'count': v, 'pct': tc_pct[k]})
for k, v in sp_dist.items():
    flag_rows.append({'flag': 'specificity', 'value': k, 'count': v, 'pct': sp_pct[k]})
for k, v in j_dist.items():
    flag_rows.append({'flag': 'judgement', 'value': k, 'count': v, 'pct': j_pct[k]})
flag_df = pd.DataFrame(flag_rows)
save_table(flag_df, 'flag_distributions.csv')
display(flag_df)